# Table Tennis Stroke Classification Pipeline
## 5-Category Framework

### Categories:
1. **Serve** (serev)
2. **Forehand Chop**
3. **Backhand Chop**
4. **Forehand Smash**
5. **Backhand Smash**

### Data Structure:
- **Videos 1-6**: Match Rallies
- **Videos 7-14**: Training Drills

In [ ]:
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
from pathlib import Path
import json
from collections import defaultdict, Counter

# Define paths
BASE_PATH = Path(r'c:\DD\TTS_SDS\TEST2\data')
CSV_PATH = BASE_PATH / 'annotations' / 'csv'
XML_PATH = BASE_PATH / 'annotations' / 'xml'

# Define the 5 categories
CATEGORIES = {
    'Serve': ['serev'],
    'Forehand Chop': ['forehand_chop'],
    'Backhand Chop': ['backhand_chop'],
    'Forehand Smash': ['forehand_smash'],
    'Backhand Smash': ['backhand_smash']
}

# Video categorization
VIDEO_TYPES = {
    'match_rallies': list(range(1, 7)),      # Videos 1-6
    'training_drills': list(range(7, 15))     # Videos 7-14
}

print("✓ Paths and categories defined")
print(f"  CSV Path: {CSV_PATH}")
print(f"  XML Path: {XML_PATH}")
print(f"\n  Categories: {list(CATEGORIES.keys())}")
print(f"  Videos 1-6: Match Rallies")
print(f"  Videos 7-14: Training Drills")

## Load and Classify Data

In [ ]:
def load_csv_annotations(video_num):
    """Load CSV annotation file for a video"""
    csv_file = CSV_PATH / f'TTVideo_{video_num}.csv'
    if csv_file.exists():
        return pd.read_csv(csv_file)
    return None

def classify_stroke(stroke_type):
    """Classify a stroke into one of the 5 categories"""
    for category, stroke_list in CATEGORIES.items():
        if stroke_type in stroke_list:
            return category
    return 'Unknown'

def get_video_type(video_num):
    """Determine if video is match rally or training drill"""
    if video_num in VIDEO_TYPES['match_rallies']:
        return 'Match Rally'
    elif video_num in VIDEO_TYPES['training_drills']:
        return 'Training Drill'
    return 'Unknown'

# Load and classify all videos
all_data = []
category_distribution = defaultdict(lambda: defaultdict(int))

for video_num in range(1, 15):
    df = load_csv_annotations(video_num)
    if df is not None:
        video_type = get_video_type(video_num)
        
        # Add classification
        df['category'] = df['stroke_type'].apply(classify_stroke)
        df['video_num'] = video_num
        df['video_type'] = video_type
        
        # Count categories
        for category in df['category']:
            category_distribution[video_type][category] += 1
        
        all_data.append(df)

# Combine all data
combined_df = pd.concat(all_data, ignore_index=True)

print(f"✓ Loaded and classified {len(combined_df)} strokes from {len(all_data)} videos")
print(f"\nDataframe shape: {combined_df.shape}")
print(f"\nFirst few rows:")
print(combined_df[['video_num', 'stroke_type', 'category', 'video_type']].head(10))

## Category Distribution Analysis

In [ ]:
# Overall category distribution
print("="*60)
print("OVERALL CATEGORY DISTRIBUTION")
print("="*60)
overall_dist = combined_df['category'].value_counts()
print(overall_dist)
print(f"\nTotal strokes: {overall_dist.sum()}")

# By video type
print("\n" + "="*60)
print("CATEGORY DISTRIBUTION BY VIDEO TYPE")
print("="*60)

for video_type in ['Match Rally', 'Training Drill']:
    print(f"\n{video_type}:")
    print("-" * 40)
    subset = combined_df[combined_df['video_type'] == video_type]
    dist = subset['category'].value_counts()
    print(dist)
    print(f"Total: {dist.sum()} strokes")

## Per-Video Category Analysis

In [ ]:
# Create detailed per-video summary
video_summary = []

for video_num in range(1, 15):
    video_data = combined_df[combined_df['video_num'] == video_num]
    if len(video_data) > 0:
        video_type = video_data['video_type'].iloc[0]
        category_counts = video_data['category'].value_counts().to_dict()
        
        row = {
            'Video': f'TTVideo_{video_num}',
            'Type': video_type,
            'Total_Strokes': len(video_data),
        }
        
        # Add count for each category (0 if not present)
        for category in CATEGORIES.keys():
            row[category] = category_counts.get(category, 0)
        
        video_summary.append(row)

summary_df = pd.DataFrame(video_summary)
print("PER-VIDEO CATEGORY BREAKDOWN:")
print("="*100)
print(summary_df.to_string(index=False))

## Check for Data Completeness

In [ ]:
print("\n" + "="*60)
print("VIDEOS WITH ALL 5 CATEGORIES")
print("="*60)

videos_with_all_categories = []

for video_num in range(1, 15):
    video_data = combined_df[combined_df['video_num'] == video_num]
    if len(video_data) > 0:
        categories_present = set(video_data['category'].unique())
        required_categories = set(CATEGORIES.keys())
        
        has_all = required_categories.issubset(categories_present)
        missing = required_categories - categories_present
        
        video_type = video_data['video_type'].iloc[0]
        
        if has_all:
            videos_with_all_categories.append(f'TTVideo_{video_num}')
            status = "✓ COMPLETE"
        else:
            status = f"✗ MISSING: {', '.join(missing)}"
        
        print(f"TTVideo_{video_num} ({video_type:15}): {status}")

print(f"\n→ {len(videos_with_all_categories)} videos have all 5 categories")
print(f"  Videos: {', '.join(videos_with_all_categories)}")

## Export Results

In [ ]:
import os

# Create output directory
OUTPUT_PATH = Path(r'c:\DD\TTS_SDS\TEST2\data\classification_results')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Export 1: Complete classified dataset
output_csv = OUTPUT_PATH / 'all_strokes_classified.csv'
combined_df.to_csv(output_csv, index=False)
print(f"✓ Exported complete dataset: {output_csv}")

# Export 2: Video summary
output_summary = OUTPUT_PATH / 'video_category_summary.csv'
summary_df.to_csv(output_summary, index=False)
print(f"✓ Exported video summary: {output_summary}")

# Export 3: Metadata JSON
metadata = {
    'categories': CATEGORIES,
    'video_types': VIDEO_TYPES,
    'total_videos': len(all_data),
    'total_strokes': len(combined_df),
    'category_distribution': combined_df['category'].value_counts().to_dict(),
    'videos_with_all_categories': videos_with_all_categories
}

output_metadata = OUTPUT_PATH / 'classification_metadata.json'
with open(output_metadata, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Exported metadata: {output_metadata}")

## New Data Classification Template

In [ ]:
def classify_new_video(video_csv_path, video_num, video_type='Unknown'):
    """
    Classify a new video's strokes into the 5 categories.
    
    Parameters:
    -----------
    video_csv_path : str
        Path to the CSV file with stroke annotations
    video_num : int
        Video number for tracking
    video_type : str
        'Match Rally' or 'Training Drill'
    
    Returns:
    --------
    dict : Classification results for the video
    """
    
    # Load CSV
    df = pd.read_csv(video_csv_path)
    
    # Classify
    df['category'] = df['stroke_type'].apply(classify_stroke)
    df['video_num'] = video_num
    df['video_type'] = video_type
    
    # Get statistics
    categories_present = set(df['category'].unique())
    required_categories = set(CATEGORIES.keys())
    missing_categories = required_categories - categories_present
    has_all_categories = len(missing_categories) == 0
    
    # Create result
    result = {
        'video_num': video_num,
        'video_type': video_type,
        'total_strokes': len(df),
        'category_distribution': df['category'].value_counts().to_dict(),
        'has_all_5_categories': has_all_categories,
        'missing_categories': list(missing_categories),
        'classified_data': df
    }
    
    return result

# Example usage template
example_code = """
# Usage for new video (e.g., TTVideo_15):
new_video_result = classify_new_video(
    video_csv_path=r'c:\\path\\to\\TTVideo_15.csv',
    video_num=15,
    video_type='Match Rally'  # or 'Training Drill'
)

# Check results
print(f"Total strokes: {new_video_result['total_strokes']}")
print(f"Categories: {new_video_result['category_distribution']}")
print(f"Has all 5 categories: {new_video_result['has_all_5_categories']}")
if new_video_result['missing_categories']:
    print(f"Missing: {new_video_result['missing_categories']}")

# Save classified data
new_video_result['classified_data'].to_csv(
    OUTPUT_PATH / 'TTVideo_15_classified.csv',
    index=False
)
"""

print("NEW DATA CLASSIFICATION TEMPLATE:")
print("=" * 60)
print(example_code)

## Summary & Key Findings

In [ ]:
print("\n" + "="*70)
print("CLASSIFICATION PIPELINE SUMMARY")
print("="*70)

print(f"\n📊 DATASET STATISTICS:")
print(f"   • Total Videos Analyzed: {len(all_data)}")
print(f"   • Total Strokes: {len(combined_df)}")
print(f"   • Match Rallies (1-6): {len(combined_df[combined_df['video_type']=='Match Rally'])} strokes")
print(f"   • Training Drills (7-14): {len(combined_df[combined_df['video_type']=='Training Drill'])} strokes")

print(f"\n🎯 5 CATEGORIES:")
for i, category in enumerate(CATEGORIES.keys(), 1):
    count = len(combined_df[combined_df['category'] == category])
    pct = (count / len(combined_df)) * 100
    print(f"   {i}. {category:20} : {count:4} strokes ({pct:5.1f}%)")

print(f"\n✓ VIDEOS WITH ALL 5 CATEGORIES: {len(videos_with_all_categories)}")
for vid in videos_with_all_categories:
    print(f"   • {vid}")

print(f"\n📁 OUTPUT FILES CREATED:")
print(f"   • all_strokes_classified.csv")
print(f"   • video_category_summary.csv")
print(f"   • classification_metadata.json")
print(f"   Location: {OUTPUT_PATH}")

print(f"\n→ Use classify_new_video() function to classify new data!")